# Vidal et al. (2019) Case 1
Independent scratch training for Panasonic 18650PF, LG HG2, and Samsung 30T: `full_train -> test`. SOC v2 is a cycler-derived benchmark reference, not independently measured ground truth. Sequence length, min-max normalisation, and shuffle are reconstruction assumptions. This notebook starts with smoke mode; it does not launch 515 epochs automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, subprocess, sys
from datetime import datetime, timezone
REPO_ROOT = Path('/content/battery-soc-transfer-benchmark')
DRIVE_ROOT = Path('/content/drive/MyDrive/battery_soc_benchmark')
VIDAL_V2_ROOT = DRIVE_ROOT / 'data/labelled/vidal_sequence_v2'
CASE1_OUTPUT_ROOT = DRIVE_ROOT / 'outputs/vidal_case1'
LOCAL_CACHE_ROOT = Path('/content/battery_soc_cache/vidal_sequence_v2')
assert REPO_ROOT.is_dir(), f'Missing repo: {REPO_ROOT}'
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Paths configured:', VIDAL_V2_ROOT, CASE1_OUTPUT_ROOT, sep='\n')

## Git and Drive preflight

In [ ]:
from battery_benchmark.data.vidal_sequence_v2_dataset import validate_case1_roots
commit = subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO_ROOT, text=True).strip()
dirty = bool(subprocess.check_output(['git','status','--porcelain'], cwd=REPO_ROOT, text=True).strip())
checks = validate_case1_roots(VIDAL_V2_ROOT)
display(checks)
assert (checks.status == 'PASS').all(), 'Required Drive input is missing'
CASE1_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
test_path = CASE1_OUTPUT_ROOT / '.case1_drive_write_test.json'
payload = {'utc_timestamp': datetime.now(timezone.utc).isoformat(), 'git_commit': commit, 'git_dirty': dirty, 'repository': REPO_ROOT.name, 'marker': 'battery_soc_case1_drive_test'}
encoded = json.dumps(payload, sort_keys=True).encode()
test_path.write_bytes(encoded)
assert test_path.read_bytes() == encoded
print('Drive mount: PASS; Output create: PASS; Test write/readback: PASS')
print('SHA-256:', hashlib.sha256(encoded).hexdigest())
test_path.unlink(); assert not test_path.exists(); print('Cleanup: PASS')
print('Commit:', commit, 'dirty:', dirty)

## Inventory and QC (read-only)

In [ ]:
from battery_benchmark.data.vidal_sequence_v2_dataset import VidalSequenceV2Dataset
for dataset in ['panasonic_18650pf','lg_hg2','samsung_30t']:
    train = VidalSequenceV2Dataset(VIDAL_V2_ROOT, dataset, 'full_train')
    test = VidalSequenceV2Dataset(VIDAL_V2_ROOT, dataset, 'test')
    print(dataset, {'train_records': len(train.records), 'test_records': len(test.records), 'train_status': train.inventory.label_qc_status.value_counts().to_dict(), 'test_status': test.inventory.label_qc_status.value_counts().to_dict()})
    display(train.inventory.head())

## Development smoke run
Core loading, normalisation, windowing, training, evaluation, checkpointing, prediction export, and plotting live in `src/battery_benchmark/experiments/vidal_case1.py`. Results below are development-only and must not enter the formal summary.

In [ ]:
!python {REPO_ROOT}/scripts/run_vidal_case1.py \
  --dataset all \
  --config {REPO_ROOT}/configs/experiments/vidal_case1_smoke.yaml \
  --input-root {VIDAL_V2_ROOT} \
  --output-root {CASE1_OUTPUT_ROOT} \
  --cache-root {LOCAL_CACHE_ROOT} \
  --run-mode smoke \
  --device auto \
  --max-train-windows 32

## Inspect saved results

In [ ]:
from IPython.display import Image, display
runs = sorted((CASE1_OUTPUT_ROOT / '_smoke').glob('*/*/*'))
for run in runs[-3:]:
    print('Run:', run)
    print(json.loads((run/'metrics.json').read_text()))
    print(json.loads((run/'scaler.json').read_text()))
    for image in sorted((run/'figures').glob('*.png')):
        print('Saved:', image); display(Image(filename=str(image)))

## Formal run reminder
After reviewing the smoke artifacts, commit the code yourself and rerun from a clean checkout with the dataset-specific formal config. Formal mode requires 515 epochs, all training windows, no early stopping, and refuses dirty Git state by default.